**Target Audience:**
This analysis of Chicago 2025 crime data is intended for public safety officials, urban planners, and residents, providing insights into crime patterns across neighborhoods and times to support informed decisions on policing, safety, and community planning.

**Relevance and Motivation:**
The analysis focuses on crime trends in 2025 by type, location, and time, helping stakeholders identify hotspots, high-risk hours, and areas needing targeted interventions. The goal is to provide actionable insights for improving safety and allocating resources effectively.

# Load data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

df = pd.read_csv("chicago_crime_2025.csv")

df["date"] = pd.to_datetime(df["date"])
df["month"] = df["date"].dt.month
df["hour"] = df["date"].dt.hour

# Data Quality Checks

**Missing Values Table (Aggregate Table #1)**

In [ ]:
missing_table = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percent": (df.isnull().sum()/len(df))*100
}).sort_values("missing_percent", ascending=False)

missing_table

**Data validity checks**

In [ ]:
df.describe()

In [ ]:
df.groupby("primary_type").size()

In [ ]:
df.groupby("month").size()

In [ ]:
df.nunique().sort_values(ascending=False)

**Identify Categorical Columns**

In [ ]:
categorical_cols = [
    "primary_type",       # type of crime
    "location_description", 
    "district",
    "ward",
    "community_area",
    "arrest",             # boolean, but categorical
    "domestic"
]
unique_counts = df[categorical_cols].nunique().sort_values(ascending=False)
unique_counts

**Show Top Categories for Each Column**

In [ ]:
top_values = {col: df[col].value_counts().head(5) for col in categorical_cols}
top_values

# Groupby Analysis

**Groupby #1 — Crimes by Type (Aggregate Table #2)**

In [ ]:
crime_type_table = df.groupby("primary_type").size().sort_values(ascending=False)

crime_type_table.head(15)

**Groupby #2 — Crimes by Month (Aggregate Table #3)**

In [ ]:
monthly_crime = df.groupby("month").size()

monthly_crime

**Groupby #3 — Arrest Rate by Crime Type**

In [ ]:
arrest_rate = df.groupby("primary_type")["arrest"].mean().sort_values(ascending=False)

arrest_rate.head(15)

# APPLY / MAP USAGE

**Map Example — Month Names**

In [ ]:
month_map = {
1:"Jan",2:"Feb",3:"Mar",4:"Apr",
5:"May",6:"Jun",7:"Jul",8:"Aug",
9:"Sep",10:"Oct",11:"Nov",12:"Dec"
}

df["month_name"] = df["month"].map(month_map)
df.head()

**Apply Example — Time of Day Category**

In [ ]:
def time_of_day(hour):
    if hour < 6:
        return "Late Night"
    elif hour < 12:
        return "Morning"
    elif hour < 18:
        return "Afternoon"
    else:
        return "Evening"

df["time_of_day"] = df["hour"].apply(time_of_day)
df.head()

# CHARTS

**Chart 1 — Top Crime Types**

In [ ]:
plt.figure(figsize=(10,6))

df["primary_type"].value_counts().head(10).plot(
    kind="bar",
    color="steelblue"
)

plt.title("Top 10 Crime Types in Chicago (2025)")
plt.xlabel("Crime Type")
plt.ylabel("Number of Incidents")

plt.xticks(rotation=45)

plt.show()

**Chart 2 — Crime Trend by Month**

In [ ]:
plt.figure(figsize=(10,6))

monthly_crime.plot(
    kind="line",
    marker="o",
    color="darkred"
)

plt.title("Crime Trend by Month in 2025")
plt.xlabel("Month")
plt.ylabel("Number of Crimes")

plt.show()

**Chart 3 — Crime Distribution by Hour**

In [ ]:
plt.figure(figsize=(10,6))

sns.countplot(
    x="hour",
    data=df,
    color="teal"
)

plt.title("Crime Distribution by Hour of Day")
plt.xlabel("Hour")
plt.ylabel("Crime Count")

plt.show()

**Chart 4 — Arrest Rate by Crime Type**

In [ ]:
plt.figure(figsize=(10,6))

arrest_rate.head(10).plot(
    kind="barh",
    color="orange"
)

plt.title("Crime Types with Highest Arrest Rate")
plt.xlabel("Arrest Rate")
plt.ylabel("Crime Type")

plt.show()

**Crime by Time of Day (Using apply variable)**

In [ ]:
time_of_day_crime = df.groupby("time_of_day").size()

time_of_day_crime

In [ ]:
plt.figure(figsize=(8,5))

time_of_day_crime.plot(
    kind="bar",
    color="purple"
)

plt.title("Crime Distribution by Time of Day")
plt.xlabel("Time of Day")
plt.ylabel("Number of Crimes")

plt.show()

**Day-of-Week vs Hour Crime Heatmap**

In [ ]:
df["day_of_week"] = df["date"].dt.day_name()

day_order = [
"Monday","Tuesday","Wednesday",
"Thursday","Friday","Saturday","Sunday"
]

df["day_of_week"] = pd.Categorical(df["day_of_week"], categories=day_order, ordered=True)

crime_heatmap = df.groupby(["day_of_week","hour"]).size().unstack()

plt.figure(figsize=(12,6))

sns.heatmap(
    crime_heatmap,
    cmap="Reds"
)

plt.title("Crime Frequency by Hour and Day of Week (2025)")
plt.xlabel("Hour of Day")
plt.ylabel("Day of Week")

plt.show()